## Primer zadatka klasifikacije - kNN

In [ ]:
# Import potrebnih biblioteka
import pandas as pd
import numpy as np
from scipy.stats import shapiro
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score,precision_score,recall_score,f1_score


In [ ]:
# Učitavamo podatke (navodimo koje vrednosti ćemo smatrati nedostajućim: '-', ' ', '')
df = pd.read_csv("./data/spotify.csv", na_values=['-', ' ', ''])

display(df.head())
df.info()

## Kreiranje izlazne varijable

In [ ]:
# Proveravamo da li varijabla streams ima nedostajuće vrednosti jer na osnovu nje kreiramo izlaznu varijablu
print("Broj nedostajućih vrednosti varijable streams:", df['streams'].isna().sum())
# Nema nedostajućih vrednosti

# Računamo treći kvartil
streams_3Q = df['streams'].quantile(0.75)
print(streams_3Q)

# Kreiramo izlaznu varijablu HighStreams na osnovu vrednosti varijable streams
df['HighStreams'] = np.where(df['streams'] > streams_3Q, 'Yes', 'No')
df['HighStreams'] = df['HighStreams'].map({'Yes': 1, 'No': 0})

# Odmah izbacujemo varijablu streams jer smo je koristili za kreiranje izlazne varijable
df.drop(columns='streams', inplace=True)

## Zamena nedostajućih vrednosti

In [ ]:
# TODO

## Procena značajnosti atributa za uključivanje u model

In [ ]:
# TODO

## Skaliranje numeričkih varijabli


In [ ]:
print(df.describe())
print(df.info())

In [ ]:
numeric_vars = ['released_year', 'in_spotify_playlists', 'in_spotify_charts', 'in_apple_playlists', 'in_apple_charts']

Proveravamo da li numeričke varijable imaju autlajere kako bismo odlučili da li ćemo koristiti `normalizaciju` ili `standardizaciju` za skaliranje podataka. Ako varijable nemaju autlajere, koristićemo normalizaciju, a ako imaju, koristićemo standardizaciju.


In [ ]:
# Funkcija za izračunavanje broja autlajera
def count_outliers(variable):
    q1 = variable.quantile(0.25)
    q3 = variable.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return ((variable < lower) | (variable > upper)).sum()

outliers_per_column = df[numeric_vars].apply(count_outliers)
print("Broj autlajera po kolonama:\n", outliers_per_column)
# Sve varijable imaju autlajere, tako da ćemo koristiti standardizaciju

In [ ]:
# Proveravamo da li varijable imaju normalnu raspodelu kako bismo odlučili na koji način da izvršimo standardizaciju
shapiro_results = df[numeric_vars].apply(lambda x: shapiro(x)[1])
print("Shapiro-Wilk p-vrednosti:\n", shapiro_results)

# Sve varijable imaju p-vrednosti < 0.05, što znači da nemaju normalnu raspodelu
# Zbog toga ćemo ih standardizovati pomoću RobustScaler-a, koji koristi median i IQR (interkvartilni opseg, razlika između 3. i 1. kvartila)

In [ ]:
# Standardizujemo numeričke varijable pomoću RobustScaler-a
robust_scaler = RobustScaler()
df_st = pd.DataFrame() # Kreiramo prazan DataFrame u koji ćemo ubaciti sve vrednosti pripremljene za KNN
df_st[numeric_vars] = robust_scaler.fit_transform(df[numeric_vars])

### Transformacija binarnih i kategorijskih varijabli u numeričke


In [ ]:
print(df.info())
# Jedina varijabla koja nije numerička je mode

In [ ]:
print(df['mode'].unique())
# Varijabla mode ima samo dve vrednosti ('Major' i 'Minor'), binarna je i možemo je mapirati u brojeve (npr. 0 i 1)
df_st['mode'] = df['mode'].map({'Major': 0, 'Minor': 1})

In [ ]:
# Dodajemo i izlaznu varijablu HighStreams u standardizovani DataFrame
df_st['HighStreams'] = df['HighStreams']

### Podela na trening i test skupove

In [ ]:
# Delimo podatke na trening (80%) i test (20%) skup

X = df_st.drop(columns='HighStreams')
y = df_st['HighStreams']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### Izračunavanje optimalne vrednosti za parametar `k`

In [ ]:
knn = KNeighborsClassifier()

# Koristimo neparne vrednosti za k (broj suseda) u opsegu od 3 do 25 kako bi uvek postojala dominantna klasa
param_grid = {'n_neighbors': list(range(3, 26, 2))}

# Primenjujemo kros validaciju sa 10 iteracija
cv = GridSearchCV(estimator=knn,
                  param_grid=param_grid,
                  cv=10,
                  scoring='accuracy',
                  verbose = 1,
                  n_jobs = -1)
cv.fit(X_train, y_train)

best_k = cv.best_params_['n_neighbors']
print("Optimalna vrednost za k:", best_k)
# Dobili smo da nam je optimalna vrednost za parametar k=11

### Kreiranje KNN klasifikatora na osnovu dobijene vrednosti za k


In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k)
knn.fit(X_train, y_train)

In [ ]:
y_pred = knn.predict(X_test)

### Kreiranje matrice konfuzije

In [ ]:
cm_knn = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm_knn,
                              display_labels=['No', 'Yes'])
disp.plot(cmap=plt.cm.Blues)
plt.show()

#### Interpretacija matrice konfuzije

- **True positive (TP)** – za 34 pesama smo predvideli da će imati veliki broj puštanja i one zaista imaju veliki broj puštanja
- **True negative (TN)** – za 137 pesama smo predvideli da neće imati veliki broj puštanja i one zaista nemaju veliki broj puštanja
- **False positive (FP)** – za 6 pesama smo predvideli da će imati veliki broj puštanja, a one zapravo nemaju veliki broj puštanja
- **False negative (FN)** – za 14 pesama smo predvideli da neće imati veliki broj puštanja, a one zapravo imaju veliki broj puštanja

Na glavnoj dijagonali matrice (34 i 137) nalaze se tačne predikcije, dok se na sporednoj dijagonali (6 i 14) nalaze pogrešne predikcije.


### Evaluacione metrike

4 metrike koje se najčešće koriste za procenu uspešnosti klasifikatora su:

- **Accuracy (tačnost)** – procenat tačnih predikcija (i pozitivnih i negativnih) u odnosu na sve observacije
- **Precision (preciznost)** – procenat observacija za koje smo predvideli da su pozitivne i koje zaista jesu pozitivne
- **Recall (odziv)** – procenat observacija koje su stvarno pozitivne i koje smo mi tačno predvideli kao pozitivne
- **F1 score** – mera za balansiranje vrednosti precision-a i recall-a


In [ ]:
def compute_eval_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, pos_label=1)
    recall = recall_score(y_true, y_pred, pos_label=1)
    f1 = f1_score(y_true, y_pred, pos_label=1)
    return {
        'accuracy': accuracy,
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1)
    }

In [ ]:
knn_eval = compute_eval_metrics(y_test, y_pred)
print(knn_eval)

Tumačenje metrika u kontekstu problema koji se rešava u zadatku:
- **Accuracy** – za 89.53% pesama smo tačno predvideli da li će imati veliki broj puštanja ili ne
- **Precision** – od svih pesama za koje smo predvideli da će imati veliki broj puštanja, 85% zaista ima veliki broj puštanja
- **Recall** – od svih pesama koje zaista imaju veliki broj puštanja, mi smo tačno predvideli 70.83% njih
- **F1 score** – mera koja balansira preciznost i odziv, i u našem slučaju iznosi 0.77. Može se reći da naš model uspešno balansira između sposobnosti da prepozna većinu pesama koje zaista imaju veliki broj puštanja i toga da ne označava previše pesama kao pesme sa velikim brojem puštanja kada one to zapravo nisu.


**Napomena:** Date interpretacije u ovom rešenju predstavljaju samo jedan od mogućih načina formulacije odgovora na zahtev zadatka.
